## Антифрод-правила

Так как в датасете нет явной метки фрода, антифрод-правила сформулированы как экспертные интерпретируемые правила для редких рискованных сегментов.

Средний уровень дефолта в train-выборке составляет **8.07%**. Каждое правило покрывает не более **2%** train-выборки и имеет default rate выше среднего.

| № | Название правила | Условие | Доля train-выборки | Default rate | Интерпретация |
|---|---|---|---:|---:|---|
| 1 | `AF001_low_external_score_pair` | `EXT_SOURCE_2 < 0.20` и `EXT_SOURCE_3 < 0.20` | 0.87% | 37.15% | Два внешних скоринговых источника одновременно дают очень низкую оценку клиента. |
| 2 | `AF002_young_high_credit_to_income` | возраст клиента `< 25 лет` и `credit_to_income_ratio > 3.0` | 1.43% | 14.00% | Молодой клиент запрашивает кредит, который более чем в 3 раза превышает годовой доход. |
| 3 | `AF003_low_skill_laborers` | `OCCUPATION_TYPE == "Low-skill Laborers"` | 0.68% | 17.15% | Узкая профессиональная группа с повышенным уровнем дефолта в обучающей выборке. |

Если для заявки выполняется хотя бы одно из этих правил, заявка считается попавшей в антифрод-фильтр и исключается из обучающей выборки модели.

При проверке заявки правила применяются последовательно:

1. `AF001_low_external_score_pair`
2. `AF002_young_high_credit_to_income`
3. `AF003_low_skill_laborers`

Метод `AntifraudService.check()` возвращает название первого сработавшего правила. Если ни одно правило не выполнено, возвращается строка `"accepted"`.


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

import pandas as pd

from app.antifraud import AntifraudService

df = pd.read_parquet(PROJECT_ROOT / "artifacts" / "hw6_core" / "application_scoring_features.parquet")
train = df[df["DATASET_SOURCE"].eq("train")].copy()
baseline = train["TARGET"].mean()

baseline


0.08072881945686496

In [3]:
service = AntifraudService()

rule_result = train.apply(service.check, axis=1)
summary = (
    train.assign(antifraud_rule=rule_result)
    .query("antifraud_rule != 'accepted'")
    .groupby("antifraud_rule")
    .agg(rows=("TARGET", "size"), default_rate=("TARGET", "mean"))
    .reset_index()
)

summary["share"] = summary["rows"] / len(train)
summary["lift"] = summary["default_rate"] / baseline
summary[["antifraud_rule", "rows", "share", "default_rate", "lift"]]


,antifraud_rule,rows,share,default_rate,lift
0,AF001_low_external_score_pair,2662,0.008657,0.371525,4.602138
1,AF002_young_high_credit_to_income,4286,0.013938,0.133691,1.656052
2,AF003_low_skill_laborers,2020,0.006569,0.170297,2.109495


In [4]:
excluded_mask = rule_result.ne("accepted")

pd.DataFrame(
    [
        {
            "segment": "excluded_by_antifraud",
            "rows": int(excluded_mask.sum()),
            "share": excluded_mask.mean(),
            "default_rate": train.loc[excluded_mask, "TARGET"].mean(),
        },
        {
            "segment": "kept_for_training",
            "rows": int((~excluded_mask).sum()),
            "share": (~excluded_mask).mean(),
            "default_rate": train.loc[~excluded_mask, "TARGET"].mean(),
        },
    ]
)


,segment,rows,share,default_rate
0,excluded_by_antifraud,8968,0.029163,0.212533
1,kept_for_training,298543,0.970837,0.076770
